In [3]:


import random, json, csv
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

SEED=26001
BATCH=128
EPOCHS=20
LR=0.001
OUT=Path("q1_results")
OUT.mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])
dataset=datasets.CIFAR10("./data",train=True,download=True,transform=transform)
test_dataset=datasets.CIFAR10("./data",train=False,download=True,transform=transform)
labels=np.asarray(dataset.targets)
train_idx,val_idx=train_test_split(np.arange(len(labels)),test_size=5000,
                                  stratify=labels,random_state=SEED)
train_ds=Subset(dataset,train_idx)
val_ds=Subset(dataset,val_idx)
train_loader=DataLoader(train_ds,batch_size=BATCH,shuffle=True,num_workers=0)
val_loader=DataLoader(val_ds,batch_size=BATCH,shuffle=False,num_workers=0)
test_loader=DataLoader(test_dataset,batch_size=BATCH,shuffle=False,num_workers=0)

class MLP(nn.Module):
    def __init__(self,dropout=0.0,large=False):
        super().__init__()
        if large:
            sizes=[3072,1024,1024,512,10]
        else:
            sizes=[3072,512,256,10]
        layers=[]
        for i in range(len(sizes)-2):
            layers.extend([nn.Linear(sizes[i],sizes[i+1]),nn.ReLU()])
            if dropout>0:
                layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(sizes[-2],sizes[-1]))
        self.net=nn.Sequential(*layers)
    def forward(self,x):
        return self.net(x.flatten(1))

def nparams(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

@torch.no_grad()
def evaluate(model,loader):
    model.eval()
    loss_fn=nn.CrossEntropyLoss()
    total=correct=loss_sum=0
    for x,y in loader:
        x,y=x.to(device),y.to(device)
        logits=model(x)
        loss=loss_fn(logits,y)
        loss_sum+=loss.item()*y.size(0)
        correct+=(logits.argmax(1)==y).sum().item()
        total+=y.size(0)
    return loss_sum/total,correct/total

def train(model,name):
    model.to(device)
    opt=torch.optim.Adam(model.parameters(),lr=LR)
    loss_fn=nn.CrossEntropyLoss()
    hist={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]}
    for epoch in range(EPOCHS):
        model.train()
        total=correct=loss_sum=0
        for x,y in train_loader:
            x,y=x.to(device),y.to(device)
            opt.zero_grad()
            logits=model(x)
            loss=loss_fn(logits,y)
            loss.backward()
            opt.step()
            loss_sum+=loss.item()*y.size(0)
            correct+=(logits.argmax(1)==y).sum().item()
            total+=y.size(0)
        tr_loss,tr_acc=loss_sum/total,correct/total
        val_loss,val_acc=evaluate(model,val_loader)
        hist["train_loss"].append(tr_loss); hist["train_acc"].append(tr_acc)
        hist["val_loss"].append(val_loss); hist["val_acc"].append(val_acc)
        print(f"{name} Epoch {epoch+1}/{EPOCHS}: train acc={tr_acc:.4f}, val acc={val_acc:.4f}")
    test_loss,test_acc=evaluate(model,test_loader)
    hist["test_loss"]=test_loss; hist["test_acc"]=test_acc
    hist["parameters"]=nparams(model)
    with open(OUT/f"{name}_history.json","w") as f: json.dump(hist,f)
    torch.save(model.state_dict(),OUT/f"{name}.pt")
    plot(hist,name)
    return hist

def plot(h,name):
    plt.figure()
    plt.plot(h["train_acc"],label="Train accuracy")
    plt.plot(h["val_acc"],label="Validation accuracy")
    plt.xlabel("Epoch");plt.ylabel("Accuracy");plt.legend();plt.grid(True)
    plt.savefig(OUT/f"{name}_accuracy.png",dpi=200);plt.close()
    plt.figure()
    plt.plot(h["train_loss"],label="Train loss")
    plt.plot(h["val_loss"],label="Validation loss")
    plt.xlabel("Epoch");plt.ylabel("Loss");plt.legend();plt.grid(True)
    plt.savefig(OUT/f"{name}_loss.png",dpi=200);plt.close()

# Q1(a): Same MLP architecture, only dropout changes
results=[]
for name,drop in [("Q1a_MLP_no_dropout",0.0),("Q1a_MLP_dropout30",0.30)]:
    model=MLP(dropout=drop)
    print("\n",name,"parameters:",nparams(model))
    h=train(model,name)
    results.append({"experiment":name,"parameters":h["parameters"],
                    "best_val_acc":max(h["val_acc"]),"test_acc":h["test_acc"],
                    "test_loss":h["test_loss"]})

# Q1(b): Larger MLP
name="Q1b_large_MLP"
model=MLP(large=True)
print("\n",name,"parameters:",nparams(model))
h=train(model,name)
results.append({"experiment":name,"parameters":h["parameters"],
                "best_val_acc":max(h["val_acc"]),"test_acc":h["test_acc"],
                "test_loss":h["test_loss"]})

with open(OUT/"results.csv","w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=results[0].keys())
    w.writeheader();w.writerows(results)
print("\nQ1 complete. Results and curves saved to:",OUT.resolve())

Device: cpu


100%|██████████| 170M/170M [15:32<00:00, 183kB/s]



 Q1a_MLP_no_dropout parameters: 1707274
Q1a_MLP_no_dropout Epoch 1/20: train acc=0.4155, val acc=0.4556
Q1a_MLP_no_dropout Epoch 2/20: train acc=0.4904, val acc=0.4790
Q1a_MLP_no_dropout Epoch 3/20: train acc=0.5285, val acc=0.5042
Q1a_MLP_no_dropout Epoch 4/20: train acc=0.5612, val acc=0.5094
Q1a_MLP_no_dropout Epoch 5/20: train acc=0.5896, val acc=0.5194
Q1a_MLP_no_dropout Epoch 6/20: train acc=0.6168, val acc=0.5258
Q1a_MLP_no_dropout Epoch 7/20: train acc=0.6406, val acc=0.5252
Q1a_MLP_no_dropout Epoch 8/20: train acc=0.6656, val acc=0.5214
Q1a_MLP_no_dropout Epoch 9/20: train acc=0.6906, val acc=0.5280
Q1a_MLP_no_dropout Epoch 10/20: train acc=0.7111, val acc=0.5214
Q1a_MLP_no_dropout Epoch 11/20: train acc=0.7339, val acc=0.5232
Q1a_MLP_no_dropout Epoch 12/20: train acc=0.7572, val acc=0.5168
Q1a_MLP_no_dropout Epoch 13/20: train acc=0.7726, val acc=0.5128
Q1a_MLP_no_dropout Epoch 14/20: train acc=0.7942, val acc=0.5120
Q1a_MLP_no_dropout Epoch 15/20: train acc=0.8052, val acc=

In [2]:
import shutil
from pathlib import Path

# Remove corrupted or partial downloads
data_path = Path('./data')
if data_path.exists():
    print(f"Cleaning up target directory: {data_path.resolve()}")
    shutil.rmtree(data_path)

print("Cleanup completed. Please run the original cell QZlngZzd-EyH again to download the dataset freshly and execute the code.")

Cleaning up target directory: /content/data
Cleanup completed. Please run the original cell QZlngZzd-EyH again to download the dataset freshly and execute the code.
